Implementing XGBoost

In [1]:
!pip install pyswarms

In [2]:
import torch

if torch.cuda.is_available():
    print("CUDA is available! Using GPU.")
    device = torch.device("cuda")
else:
    print("CUDA is not available. Using CPU.")
    device = torch.device("cpu")


CUDA is available! Using GPU.


In [3]:
from datasets import load_dataset
ds = load_dataset("coastalchp/ledgar")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
import pandas as pd

train_df = pd.DataFrame(ds['train'])
test_df = pd.DataFrame(ds['test'])
validation_df = pd.DataFrame(ds['validation'])

In [5]:
X_train = train_df['text']
y_train = train_df['label']

X_test = test_df['text']
y_test = test_df['label']

X_val = validation_df['text']
y_val = validation_df['label']

Applying Class Weights

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
import xgboost as xbg
from xgboost import XGBClassifier

import json
with open('class_weights.json', 'r') as f:
    class_weights = json.load(f)

class_weights = {int(k): v for k, v in class_weights.items()}

In [7]:
from pyswarms.single.global_best import GlobalBestPSO

Applying TF-IDF

In [10]:
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

### **Linear SVM with PSO Optimization**

In [11]:
from sklearn.svm import LinearSVC
import numpy as np

def objective_function(c_values, X_train, y_train, X_val, y_val, class_weights):
    """
    Objective function for PSO to optimize LinearSVC's C parameter.
    PSO minimizes, so we return the negative F1-score.
    """
    n_particles = c_values.shape[0]
    f1_scores = []

    for i in range(n_particles):
        C = c_values[i, 0] # PSO optimizes a single parameter, C

        # Ensure C is positive and within a reasonable range
        C = max(0.001, C)

        model = LinearSVC(C=C, random_state=42, multi_class='ovr', class_weight=class_weights, max_iter=500)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)

        # Use 'weighted' F1-score for multi-class classification to account for class imbalance
        score = f1_score(y_val, y_pred, average='weighted')
        f1_scores.append(score)

    # PSO minimizes, so we return the negative F1-score
    return -np.array(f1_scores)

#### **Run PSO to find optimal C for LinearSVC**

In [ ]:
n_particles = 10 # Number of particles
dim = 1          # Dimension of the search space (C parameter)
options = {'c1': 0.5, 'c2': 0.3, 'w': 0.9} # PSO hyper-parameters

# Define bounds for the C parameter (e.g., 0.001 to 100)
bounds = (np.array([0.001]), np.array([100.]))

# Initialize optimizer
optimizer = GlobalBestPSO(n_particles=n_particles, dimensions=dim, options=options, bounds=bounds)

# Perform optimization
cost, pos = optimizer.optimize(objective_function, iters=10, X_train=X_train_tfidf, y_train=y_train, X_val=X_val_tfidf, y_val=y_val, class_weights=class_weights)

best_C = pos[0]
print(f"Best C found by PSO: {best_C}")

2026-04-21 20:53:40,412 - pyswarms.single.global_best - INFO - Optimize for 10 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best:   0%|          |0/10/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


#### **Train and Evaluate Final LinearSVC Model**

In [ ]:
from sklearn.metrics import classification_report

final_svm_model = LinearSVC(C=best_C, random_state=42, multi_class='ovr', class_weight=class_weights, max_iter=2000)

X_train_full = vectorizer.fit_transform(pd.concat([X_train, X_val]))
y_train_full = pd.concat([y_train, y_val])

final_svm_model.fit(X_train_full, y_train_full)

# Transform test data using the same vectorizer fitted on the full training data
X_test_tfidf = vectorizer.transform(X_test)

# Make predictions on the test set using the transformed test data
y_pred_test = final_svm_model.predict(X_test_tfidf)

# Evaluate the final model
print("\n--- Final Linear SVM Model Performance on Test Set ---")
print(classification_report(y_test, y_pred_test))